# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [23]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [24]:
# TODO
#Creating new column
df['revenue'] = df['qty'] * df['price']
#Printing the total number of rows
print("Number of rows", len(df))
#Calculating total revenue and units
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()
print("Total revenue:",total_revenue)
print("Total units:", total_units)

Number of rows 400
Total revenue: 8520.0
Total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [25]:
# Group rows by category and add up revenue within each category
by_cat = (df.groupby('category', as_index=False)['revenue'].sum())
# Calculate each category's percentage of total revenue
by_cat['share_pct'] = (by_cat['revenue'] / df['revenue'].sum() * 100)
# Sort from highest to lowest
by_cat = by_cat.sort_values('revenue',ascending=False)

print(by_cat)



   category  revenue  share_pct
1      Food   4293.0  50.387324
2     Merch   1771.5  20.792254
0     Drink   1554.0  18.239437
3  RainGear    901.5  10.580986


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [26]:
# Grouping by vendor and calculating average revenue for each order and the number of orders
vendor_stat = (df.groupby('vendor_id').agg(avg_order_revenue=('revenue', 'mean'),
                                           order_count=('revenue', 'size')).sort_values('avg_order_revenue',ascending=False))
print(vendor_stat)

print("Highest average order revenue:", vendor_stat.index[0])

           avg_order_revenue  order_count
vendor_id                                
V-01               22.595745           94
V-18               21.750000          108
V-05               20.580645           93
V-10               20.314286          105
Highest average order revenue: V-01


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [27]:
# Sum for only merch and divide by total, then make percentage
merch_revenue = df[df['category'] == 'Merch']['revenue'].sum()
total_revenue = df['revenue'].sum()
merch_share = (merch_revenue / total_revenue) * 100
print("Merch share:", round(merch_share, 1), "%")

Merch share: 20.8 %


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [28]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
joined = pd.merge(df, vendor_names, on='vendor_id', how='left', validate='many_to_one')
print(joined)

#Validating that row count and revenue total did not change
print("Number of rows before merge:", len(df))
print("Number of rows after merge:", len(joined))
print("Total revenue before merge:", df['revenue'].sum())
print("Total revenue after merge:", joined['revenue'].sum())

#Find any vendors that don't match
unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
print("Unmatched vendors:", unmatched)

#There is one unmatched vendor so I am going to give it a label
joined['vendor_name'] = joined['vendor_name'].fillna('Unmatched Vendor')
print(joined)





    vendor_id  category  qty  price  revenue      vendor_name
0        V-10     Drink    2   24.0     48.0  Cav Merch North
1        V-18  RainGear    1   12.0     12.0              NaN
2        V-18     Drink    3    4.5     13.5              NaN
3        V-10      Food    2   12.0     24.0  Cav Merch North
4        V-18     Drink    3    7.5     22.5              NaN
..        ...       ...  ...    ...      ...              ...
395      V-18     Merch    1   12.0     12.0              NaN
396      V-01     Merch    2   24.0     48.0     Hoos Burgers
397      V-10      Food    3    7.5     22.5  Cav Merch North
398      V-18     Merch    2   24.0     48.0              NaN
399      V-10     Drink    1    7.5      7.5  Cav Merch North

[400 rows x 6 columns]
Number of rows before merge: 400
Number of rows after merge: 400
Total revenue before merge: 8520.0
Total revenue after merge: 8520.0
Unmatched vendors: ['V-18']
    vendor_id  category  qty  price  revenue       vendor_name
0      

**The unmatched vendor, and what I did about it:** _The unmatched vendor was V-18 so since there was only one I decided to just keep it but label the missing data in vendor_name with Unknown Vendor._

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [29]:
# I built a pivot tabel
pivot_table = pd.pivot_table(df, index='vendor_id', columns='category', values = 'revenue', aggfunc='sum', fill_value=0, margins=True, margins_name='Total')
print(pivot_table)

category    Drink    Food   Merch  RainGear   Total
vendor_id                                          
V-01        171.0  1338.0   373.5     241.5  2124.0
V-05        298.5   882.0   489.0     244.5  1914.0
V-10        502.5  1054.5   400.5     175.5  2133.0
V-18        582.0  1018.5   508.5     240.0  2349.0
Total      1554.0  4293.0  1771.5     901.5  8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [30]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_cat['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

_a. Since food made up for the majority of the revenue (50.4%), I would tell the vendors they should prioritize marketing of food. Also, since merch made up 20.8% of revenue, marketing and expansion of any merch lines would help with increasing profits.
b. I would say that the vendor comparison is the least trustworthy because of V-18, since it would not have been read in the vendor lookup table. We don't know if that missing information could have affected the comparison
_